# Lab 10 — Cloud Operations

## Objective

Classify a synthetic incident and gate any restart recommendation on approval.

## Introduction and concept

triage combines readiness, latency and error rate. request_restart records a simulated approval decision without operating any service.

## Real-world use case

An exam API reports increased latency and repeated database timeout messages.

## Technology stack and dependencies

Python dataclasses, simulated monitoring. Use Python 3.11 or newer and a Jupyter kernel from the environment where you installed the [assignment requirements](../requirements.txt). Start the notebook from its lab directory.

## Architecture and workflow

Health + logs + metrics → Incident triage → Recommendation → Approval decision


## Step 1 — Implement the components

triage combines readiness, latency and error rate. request_restart records a simulated approval decision without operating any service.

In [1]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Metrics:
    healthy: bool
    latency_ms: int
    errors: int
    requests: int

def triage(metrics, logs):
    if metrics.latency_ms < 0 or not 0 <= metrics.errors <= metrics.requests:
        raise ValueError("Invalid monitoring metrics")
    error_rate = metrics.errors / metrics.requests if metrics.requests else 0
    timeouts = sum("DB_TIMEOUT" in line for line in logs)
    degraded = not metrics.healthy or metrics.latency_ms >= 500 or error_rate >= 0.05
    action = "inspect_connection_pool" if timeouts and degraded else "inspect_service"
    return {"incident": degraded, "error_rate": error_rate, "timeout_logs": timeouts,
            "recommendation": action if degraded else "observe"}

def request_restart(report, approved_by=None):
    if not report["incident"]:
        return "not_needed"
    if approved_by != "on_call_operator":
        return "awaiting_operator_approval"
    return "simulated_restart_approved"

## Step 2 — Run sample inputs and inspect actual output

Compare healthy and degraded metrics. Verify only the designated operator can approve the simulated restart.

In [2]:
def main():
    healthy = triage(Metrics(True, 90, 0, 50), [])
    report = triage(Metrics(False, 725, 8, 50), ["ERROR DB_TIMEOUT", "WARN retry"])
    print("Healthy service:", healthy)
    print("Degraded service:", report)
    assert not healthy["incident"] and request_restart(healthy) == "not_needed"
    assert report["recommendation"] == "inspect_connection_pool"
    for actor in [None, "trainee", "on_call_operator"]:
        print("Simulated decision:", actor, request_restart(report, actor))
    assert request_restart(report, "trainee") == "awaiting_operator_approval"
    assert request_restart(report, "on_call_operator") == "simulated_restart_approved"
    print("PASS: health classification and approval gate")

main()

Healthy service: {'incident': False, 'error_rate': 0.0, 'timeout_logs': 0, 'recommendation': 'observe'}
Degraded service: {'incident': True, 'error_rate': 0.16, 'timeout_logs': 1, 'recommendation': 'inspect_connection_pool'}
Simulated decision: None awaiting_operator_approval
Simulated decision: trainee awaiting_operator_approval
Simulated decision: on_call_operator simulated_restart_approved
PASS: health classification and approval gate


## Testing and validation

Compare healthy and degraded metrics. Verify only the designated operator can approve the simulated restart. Assertions in the preceding cell check the demonstrated behavior and stop execution on a failed expectation.

## Limitations

Health and roles are fixtures. Live monitoring, identity verification, approvals and remediation require a real operations environment.

## How this relates to ExamPilot

Partial. ExamPilot has application failures, bounded retries and a separate expiry worker. Automated cloud remediation is outside the supplied project.

## Conclusion

Classify a synthetic incident and gate any restart recommendation on approval. Use the trace and assertions to explain which decisions the workflow makes.